## Exploratory Data Analysis

### Import Libraries

In [5]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [6]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [9]:
df = pl.read_csv(
    "../data/Account.csv", 
    ignore_errors=True, 
    truncate_ragged_lines=True
    )

df

CUST_ID,ACQUISITION_COST,INTERNET_BANKING_INDICATOR,DATE_FIRST_ACCOUNT_OPENED,DATE_LAST_ACCOUNT_OPENED,PURSUIT,PRIMARY_ADVISOR_ORGANIZATION_ID,PRIMARY_BRANCH_PROXIMITY,PRIMARY_SPOKEN_LANGUAGE,PRIMARY_WRITTEN_LANGUAGE,SATISFACTION_RATING_FROM_SURVEY,SECONDARY_ADVISOR_ID,SECONDARY_ADVISOR_ORGANIZATION_ID,SPECIAL_TERMS_INDICATOR
str,f64,bool,str,str,str,i64,i64,str,str,str,i64,i64,bool
"""CUST-417911""",62.143782,true,"""2016-07-12""","""2016-07-12""","""Capital Acquisition""",1005,342,"""English""","""English""","""neutral""",117227,1004,false
"""CUST-758898""",59.672778,true,"""2016-07-12""","""2016-07-12""","""Capital Acquisition""",1013,342,"""English""","""English""","""neutral""",117227,1004,false
"""CUST-958684""",51.088502,true,"""2017-08-15""","""2017-08-15""","""Estate Planning""",1006,40,"""English""","""English""","""satisfied""",112458,1001,false
"""CUST-124574""",60.41846,true,"""2016-08-24""","""2016-08-24""","""Increase Net Worth""",1009,431,"""English""","""English""","""very satisfied""",127139,1008,true
"""CUST-198781""",73.221376,true,"""2016-08-24""","""2016-08-24""","""Increase Net Worth""",1008,431,"""English""","""English""","""very satisfied""",127139,1008,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CUST-717296""",22.301133,true,"""2017-08-28""","""2017-08-28""","""Increase Net Worth""",1006,202,"""English""","""English""","""very satisfied""",74116,1006,true
"""CUST-780694""",81.650617,true,"""2017-08-28""","""2017-08-28""","""Increase Net Worth""",1017,202,"""English""","""English""","""very satisfied""",74116,1006,true
"""CUST-787420""",19.072539,false,"""2017-11-29""","""2017-11-29""","""Retirement Planning""",1003,426,"""English""","""English""","""very dissatisfied""",139890,1016,true


### Retrieve Number of Nulls in Each Feature

In [10]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""CUST_ID""",0
"""ACQUISITION_COST""",0
"""INTERNET_BANKING_INDICATOR""",0
"""DATE_FIRST_ACCOUNT_OPENED""",0
"""DATE_LAST_ACCOUNT_OPENED""",0
"""PURSUIT""",0
"""PRIMARY_ADVISOR_ORGANIZATION_I…",0
"""PRIMARY_BRANCH_PROXIMITY""",0
"""PRIMARY_SPOKEN_LANGUAGE""",0


### Retrieve Basic Information About DataFrame

In [11]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<36} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<36} | {dtype}")

print_schema(df)

Column                               | Data Type
------------------------------------------------------------
CUST_ID                              | String
ACQUISITION_COST                     | Float64
INTERNET_BANKING_INDICATOR           | Boolean
DATE_FIRST_ACCOUNT_OPENED            | String
DATE_LAST_ACCOUNT_OPENED             | String
PURSUIT                              | String
PRIMARY_ADVISOR_ORGANIZATION_ID      | Int64
PRIMARY_BRANCH_PROXIMITY             | Int64
PRIMARY_SPOKEN_LANGUAGE              | String
PRIMARY_WRITTEN_LANGUAGE             | String
SATISFACTION_RATING_FROM_SURVEY      | String
SECONDARY_ADVISOR_ID                 | Int64
SECONDARY_ADVISOR_ORGANIZATION_ID    | Int64
SPECIAL_TERMS_INDICATOR              | Boolean


### Display Summary Statistics for All Columns

In [12]:
summary = df.describe()
print(summary)

shape: (9, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ CUST_ID   ┆ ACQUISITI ┆ INTERNET_ ┆ … ┆ SATISFACT ┆ SECONDARY ┆ SECONDARY ┆ SPECIAL_ │
│ ---       ┆ ---       ┆ ON_COST   ┆ BANKING_I ┆   ┆ ION_RATIN ┆ _ADVISOR_ ┆ _ADVISOR_ ┆ TERMS_IN │
│ str       ┆ str       ┆ ---       ┆ NDICATOR  ┆   ┆ G_FROM_SU ┆ ID        ┆ ORGANIZAT ┆ DICATOR  │
│           ┆           ┆ f64       ┆ ---       ┆   ┆ RVE…      ┆ ---       ┆ ION…      ┆ ---      │
│           ┆           ┆           ┆ f64       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 2000      ┆ 2000.0    ┆ 2000.0    ┆ … ┆ 2000      ┆ 2000.0    ┆ 2000.0    ┆ 2000.0   │
│ null_coun ┆ 0         ┆ 0.0       ┆ 0.0       ┆ … ┆ 0         ┆ 0.0       

### Find Longest Text Length in Each Column

In [13]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

CUST_ID,DATE_FIRST_ACCOUNT_OPENED,DATE_LAST_ACCOUNT_OPENED,PURSUIT,PRIMARY_SPOKEN_LANGUAGE,PRIMARY_WRITTEN_LANGUAGE,SATISFACTION_RATING_FROM_SURVEY
u32,u32,u32,u32,u32,u32,u32
11,10,10,19,7,7,17


### Retrieve Data Types of All Columns

In [14]:
print("Column data types:\n", df.dtypes)

Column data types:
 [String, Float64, Boolean, String, String, String, Int64, Int64, String, String, String, Int64, Int64, Boolean]


### Count Unique Values in Each Column

In [15]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(55), f"{unique_counts}".ljust(6))

                             Unique values in CUST_ID : 1999  
                    Unique values in ACQUISITION_COST : 2000  
          Unique values in INTERNET_BANKING_INDICATOR : 2     
           Unique values in DATE_FIRST_ACCOUNT_OPENED : 228   
            Unique values in DATE_LAST_ACCOUNT_OPENED : 228   
                             Unique values in PURSUIT : 6     
     Unique values in PRIMARY_ADVISOR_ORGANIZATION_ID : 19    
            Unique values in PRIMARY_BRANCH_PROXIMITY : 631   
             Unique values in PRIMARY_SPOKEN_LANGUAGE : 1     
            Unique values in PRIMARY_WRITTEN_LANGUAGE : 1     
     Unique values in SATISFACTION_RATING_FROM_SURVEY : 5     
                Unique values in SECONDARY_ADVISOR_ID : 995   
   Unique values in SECONDARY_ADVISOR_ORGANIZATION_ID : 19    
             Unique values in SPECIAL_TERMS_INDICATOR : 2     


### Check Distribution of Numerical Columns

In [16]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

ACQUISITION_COST
shape: (9, 2)
┌────────────┬──────────────────┐
│ statistic  ┆ ACQUISITION_COST │
│ ---        ┆ ---              │
│ str        ┆ f64              │
╞════════════╪══════════════════╡
│ count      ┆ 2000.0           │
│ null_count ┆ 0.0              │
│ mean       ┆ 49.349437        │
│ std        ┆ 29.056267        │
│ min        ┆ -0.210121        │
│ 25%        ┆ 23.72862         │
│ 50%        ┆ 48.913863        │
│ 75%        ┆ 73.615801        │
│ max        ┆ 100.445605       │
└────────────┴──────────────────┘ 


INTERNET_BANKING_INDICATOR
shape: (9, 2)
┌────────────┬────────────────────────────┐
│ statistic  ┆ INTERNET_BANKING_INDICATOR │
│ ---        ┆ ---                        │
│ str        ┆ f64                        │
╞════════════╪════════════════════════════╡
│ count      ┆ 2000.0                     │
│ null_count ┆ 0.0                        │
│ mean       ┆ 0.503                      │
│ std        ┆ null                       │
│ min        ┆ 0.0 

### List Unique Values For Certain Features

In [17]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: CUST_ID (1999 unique values)
shape: (1_999,)
Series: 'CUST_ID' [str]
[
	"CUST-100067"
	"CUST-100282"
	"CUST-100396"
	"CUST-100400"
	"CUST-100417"
	"CUST-100655"
	"CUST-100759"
	"CUST-101785"
	"CUST-101865"
	"CUST-102995"
	"CUST-103026"
	"CUST-103108"
	"CUST-103227"
	"CUST-104309"
	"CUST-104713"
	"CUST-105166"
	"CUST-106996"
	"CUST-107207"
	…
	"CUST-993400"
	"CUST-994052"
	"CUST-994211"
	"CUST-994613"
	"CUST-994795"
	"CUST-994918"
	"CUST-994965"
	"CUST-995095"
	"CUST-996204"
	"CUST-997038"
	"CUST-997072"
	"CUST-997260"
	"CUST-997681"
	"CUST-998954"
	"CUST-998988"
	"CUST-999095"
	"CUST-999182"
]
--------------------------------------------------
Column: ACQUISITION_COST (2000 unique values)
shape: (2_000,)
Series: 'ACQUISITION_COST' [f64]
[
	-0.210121
	-0.009624
	0.133669
	0.207109
	0.241075
	0.443007
	0.469562
	0.564322
	0.579648
	0.592617
	0.625614
	0.723366
	0.730375
	0.743789
	0.745143
	0.759676
	0.893706
	0.914405
	…
	99.441134
	99.536167
	99.568318
	99.584106
	99.66162
	99.

In [18]:
def list_unique_values_over_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count > threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_over_threshold(df)

### How Many Records Remain IF I Remove Records With Any Nulls In It

In [19]:
def drop_rows_with_any_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """
    Removes all rows from a Polars DataFrame that contain any null values.
    """
    return df.drop_nulls()


drop_rows_with_any_nulls(df)

CUST_ID,ACQUISITION_COST,INTERNET_BANKING_INDICATOR,DATE_FIRST_ACCOUNT_OPENED,DATE_LAST_ACCOUNT_OPENED,PURSUIT,PRIMARY_ADVISOR_ORGANIZATION_ID,PRIMARY_BRANCH_PROXIMITY,PRIMARY_SPOKEN_LANGUAGE,PRIMARY_WRITTEN_LANGUAGE,SATISFACTION_RATING_FROM_SURVEY,SECONDARY_ADVISOR_ID,SECONDARY_ADVISOR_ORGANIZATION_ID,SPECIAL_TERMS_INDICATOR
str,f64,bool,str,str,str,i64,i64,str,str,str,i64,i64,bool
"""CUST-417911""",62.143782,true,"""2016-07-12""","""2016-07-12""","""Capital Acquisition""",1005,342,"""English""","""English""","""neutral""",117227,1004,false
"""CUST-758898""",59.672778,true,"""2016-07-12""","""2016-07-12""","""Capital Acquisition""",1013,342,"""English""","""English""","""neutral""",117227,1004,false
"""CUST-958684""",51.088502,true,"""2017-08-15""","""2017-08-15""","""Estate Planning""",1006,40,"""English""","""English""","""satisfied""",112458,1001,false
"""CUST-124574""",60.41846,true,"""2016-08-24""","""2016-08-24""","""Increase Net Worth""",1009,431,"""English""","""English""","""very satisfied""",127139,1008,true
"""CUST-198781""",73.221376,true,"""2016-08-24""","""2016-08-24""","""Increase Net Worth""",1008,431,"""English""","""English""","""very satisfied""",127139,1008,true
"""CUST-228676""",25.343252,false,"""2017-04-06""","""2017-04-06""","""Retirement Planning""",1003,504,"""English""","""English""","""very satisfied""",88379,1016,false
"""CUST-464124""",44.518334,false,"""2016-02-10""","""2016-02-10""","""Increase Net Worth""",1006,468,"""English""","""English""","""very satisfied""",84254,1005,true
"""CUST-529407""",53.714719,false,"""2016-02-10""","""2016-02-10""","""Increase Net Worth""",1005,468,"""English""","""English""","""very satisfied""",84254,1005,true
"""CUST-523002""",90.463466,true,"""2016-02-13""","""2016-02-13""","""Capital Acquisition""",1008,983,"""English""","""English""","""neutral""",114807,1002,false
